This notebook is to play around with gui code for visualizing data online

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import hydra
from hydra.utils import get_original_cwd
import os
from omegaconf import DictConfig, OmegaConf
from dataclasses import dataclass
from typing import List, Dict, Any




In [3]:
# Load config
import sys
import os
from pathlib import Path

# 

# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
print(f"home directory: {cwd}")
relative_repo_path = "GitRepos/simulation_closed_loop"

# append repo path 
sys.path.append(str(cwd / relative_repo_path))

# Import Hydra config utilities
from omegaconf import DictConfig, OmegaConf
import hydra
from hydra.utils import instantiate
from hydra.core.config_store import ConfigStore
from hydra import compose, initialize

# Initialize Hydra with the relative path to the config directory
config_path = os.path.join(relative_repo_path,"config")
print(f"Config path: {config_path}")

# Initialize Hydra
with initialize(version_base="1.3", config_path=config_path):
    # Compose the configuration
    cfg = compose(config_name="config")

# Print the config to verify it loaded correctly
print("Configuration loaded successfully:")
print(OmegaConf.to_yaml(cfg))

home directory: /gpfs01/euler/User/ssuhai
Config path: GitRepos/simulation_closed_loop/config
Configuration loaded successfully:
data_subfolders:
  day: 20250717
  experiment: 1
DJ:
  username: ssuhai
  userinfo:
    experimenter: closedlooptest
    animal_loc: 1
    region_loc: 2
    field_loc: 3
    stimulus_loc: 4
    cond1_loc: 5
    data_dir: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/updated_loop_data
  table_parameters:
    PreprocessParams:
      window_length: 60
      poly_order: 3
      non_negative: 1
      subtract_baseline: 0
      standardize: 1
    Stimulus:
      noise:
        stim_name: densenoise
        stim_family: noise
        pix_n_x: 20
        pix_n_y: 15
        skip_duplicates: true
        pix_scale_x_um: 40
        pix_scale_y_um: 40
        framerate: 5
    DNoiseTraceParams:
      dnoise_params_id: 1
      fupsample_trace: 20
      fupsample_stim: 4
      ref_time: stim
      fit_kind: gradient
      skip_duplicates: true


In [4]:
cfg.paths.repo_directory

'/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/'

In [5]:
# # hydra in notebooks behaves differently. 
# # Need to overwrite the configs dynamicaly set at runtime 
# from datetime import datetime
# output_dir = f"{cfg.paths.repo_directory}/logs/outputs/{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
# OmegaConf.update(cfg, "model_configs.paths.output_dir", output_dir)


In [26]:
cfg.model_configs.paths.output_dir

'output'

In [5]:
# 
from simulations.loop_components.dj_wrappers import DJTableHolder,Preprocessor,QualityAndTypeWrapper,STAWrapper,RandomSeedMEIWrapper

# connect populated closed loop schema

In [6]:
# create preprocessor
os.environ["DJ_SUPPORT_FILEPATH_MANAGEMENT"] = "TRUE"

dj_table_holder = DJTableHolder(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore
                data_subfolders=cfg.data_subfolders, # type: ignore


                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore

                    )

In [7]:
preprocessor = Preprocessor(dj_table_holder=dj_table_holder)

In [8]:
quality_type_analysis_wrapper = QualityAndTypeWrapper(
    dj_table_holder=dj_table_holder,)

sta_wrapper = STAWrapper(
    dj_table_holder=dj_table_holder,)

randomp_seed_mei_wrapper = RandomSeedMEIWrapper(
    dj_table_holder=dj_table_holder,
    model_configs=cfg.model_configs, )

In [9]:
# # Load config and tables

dj_table_holder.load_config()
dj_table_holder.load_tables()
# dj_table_holder.clear_tables("all")

# print(" loaded and configured successfully")
#dj_table_holder.setup()




[2025-07-31 13:29:34,709][INFO]: Connecting ssuhai@172.25.240.200:3306
[2025-07-31 13:29:34,764][INFO]: Connected ssuhai@172.25.240.200:3306


schema_name: ageuler_ssuhai_closed_loop


In [12]:
preprocessor.upload_iteration_metadata()

Scanning for experimenter: closedlooptest
	header_path: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/updated_loop_data/20200226/1
		header_name: 20200226__left.ini
		Adding: {'experimenter': 'closedlooptest', 'date': datetime.datetime(2020, 2, 26, 0, 0), 'exp_num': 1}


OpticDisk: 100%|██████████| 1/1 [00:00<00:00, 29.18it/s]


Found 4 files in 1 fields for key={'experimenter': 'closedlooptest', 'date': datetime.date(2020, 2, 26), 'exp_num': 1, 'raw_id': 1}
	Adding field: `{'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0', 'experimenter': 'closedlooptest', 'date': datetime.date(2020, 2, 26), 'exp_num': 1, 'raw_id': 1}`


Processes: 100%|██████████| 6/6 [00:07<00:00,  1.33s/it]


# GUI Components for Visualization

Let's create a basic GUI to visualize the data processed by the OpenRetinaWrapper.

In [13]:
missing_keys = dj_table_holder("RoiMask")().list_missing_field()
missing_keys
# import datetime
# test_key = {'experimenter': 'closedlooptest',
#   'date': datetime.date(2025, 7, 17),
#   'exp_num': 1,
#   'raw_id': 1,
#   'field': 'GCL1',
#   'region': 'RR',
#   'cond1': 'iter0'}
# missing_keys = [test_key]

[{'experimenter': 'closedlooptest',
  'date': datetime.date(2020, 2, 26),
  'exp_num': 1,
  'raw_id': 1,
  'field': 'GCL0',
  'region': 'LR',
  'cond1': 'iter0'}]

# modified autorois 

In [12]:
dj_table_holder("Traces")()

experimenter name of the experimenter,date date of recording,exp_num experiment number in a day,raw_id unique param set id,field string identifying files corresponding to field,region region (e.g. LR or RR),cond1 condition (pharmacological or other),stim_name Unique string identifier,cond2 condition (pharmacological or other),roi_id integer id of each ROI,trace array of raw trace,trace_t0 numerical array of trace times,trace_dt time between frames,trace_valid Are values in trace correct (1) or not (0)?,trigger_valid Are triggertimes inside trace_times (1) or not (0)?
closedlooptest,2020-02-26,1,1,GCL0,LR,iter0,densenoise,control,1,=BLOB=,0.002,0.128,1,1
closedlooptest,2020-02-26,1,1,GCL0,LR,iter0,densenoise,control,2,=BLOB=,0.002,0.128,1,1
closedlooptest,2020-02-26,1,1,GCL0,LR,iter0,densenoise,control,3,=BLOB=,0.0,0.128,1,1
closedlooptest,2020-02-26,1,1,GCL0,LR,iter0,densenoise,control,4,=BLOB=,0.008,0.128,1,1
closedlooptest,2020-02-26,1,1,GCL0,LR,iter0,densenoise,control,5,=BLOB=,0.003,0.128,1,1
closedlooptest,2020-02-26,1,1,GCL0,LR,iter0,densenoise,control,6,=BLOB=,0.006,0.128,1,1
closedlooptest,2020-02-26,1,1,GCL0,LR,iter0,densenoise,control,7,=BLOB=,0.012,0.128,1,1
closedlooptest,2020-02-26,1,1,GCL0,LR,iter0,densenoise,control,8,=BLOB=,0.006,0.128,1,1
closedlooptest,2020-02-26,1,1,GCL0,LR,iter0,densenoise,control,9,=BLOB=,0.01,0.128,1,1
closedlooptest,2020-02-26,1,1,GCL0,LR,iter0,densenoise,control,10,=BLOB=,0.013,0.128,1,1


In [15]:
from simulations.gui.integrated_autorois import InteractiveRoiCanvas
# import ipywidgets as widgets
# from ipycanvas import MultiCanvas
from IPython.display import display

In [16]:
mod_autorois = InteractiveRoiCanvas(
    dj_table_holder=dj_table_holder,
    dj_preprocessor=preprocessor,
    all_dj_wrappers=[quality_type_analysis_wrapper,sta_wrapper,randomp_seed_mei_wrapper],
    field_key=missing_keys[0],
    canvas_width=30,
    )

Load model weights for cpu from checkpoint /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/dropout_and_aug_regul.ckpt using config /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/sd_images.yaml


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

...

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

Upsampling natural spikes traces to get final responses.:   0%|          | 0/1 [00:00<?, ?it/s]

InterpolationResolutionError: ValueError raised while resolving interpolation: HydraConfig was not set
    full_key: model_configs.paths.output_dir
    object_type=dict

<Figure size 640x480 with 0 Axes>

In [ ]:
# mod_autorois.update_roi_mask_img()

In [17]:
display(mod_autorois.start_gui()) 

In [14]:
dj_table_holder("CascadeTraces")()
dj_table_holder("OpenRetinaHoeflingFormat")().delete()

[2025-07-31 13:32:12,522][INFO]: Deleting 1 rows from `ageuler_ssuhai_closed_loop`.`open_retina_hoefling_format`
--- Logging error ---
Traceback (most recent call last):
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/logging/__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
ValueError: I/O operation on closed file.
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/tornado/platform/asy

1

In [10]:
print(1+ 1)
test_dict = randomp_seed_mei_wrapper.compute_analysis()

2


Processes: 100%|██████████| 103/103 [00:00<00:00, 612.57it/s]


Adding /gpfs01/euler/data/Resources/GitHub/external_repos/Cascade/ to sys.path


/gpfs01/euler/data/Resources/GitHub/external_repos/Cascade/cascade2p/cascade.py:556: SyntaxWarning: invalid escape sequence '\d'
  noise_level = int(re.findall("_NoiseLevel_(\d+)", model_path)[0])


	YAML reader installed (version 0.18.6).


2025-07-31 13:30:24.352631: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


	Keras installed (version 3.8.0).
	Tensorflow installed (version 2.16.1).


CascadeSpikes:   0%|          | 0/1 [00:00<?, ?it/s]2025-07-31 13:30:37.153505: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


CascadeSpikes: 100%|██████████| 1/1 [00:27<00:00, 27.51s/it]


In [13]:
test_dict["online_session_1_ventral1_20200226"]["natural_spikes"]

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], dtype=float32)